# Experiment 6 — K-Means Clustering using Python

### Step 1: Load the dataset

In [ ]:
import pandas as pd

df = pd.read_csv('application_matching_exp67.csv')

features = ['cgpa', 'match_score', 'skill_gap_count', 'github_repos',
            'projects_completed', 'expected_stipend', 'applications_submitted']
X = df[features]          # true_persona and student_id are deliberately left out

print("Dataset shape:", df.shape)
print("Dataset Info:")
print(X.info())
print("\nMissing Values:")
print(X.isnull().sum())

### Step 2: Handle missing values

In [ ]:
X = X.fillna(X.median())
print("Missing values after imputation:", X.isnull().sum().sum())

### Step 3: Standardise the data

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=features)

### Step 4: Plot the Elbow Method graph

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

inertia = []
k_range = range(1, 11)
for k in k_range:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia, 'bo-', markersize=8)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.grid(True)
plt.show()

### Step 5: Apply K-Means with k = 3

In [ ]:
k = 3
kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
kmeans.fit(X_scaled)

X_scaled_df['Cluster'] = kmeans.labels_
print(f"K-Means applied with k={k}")
print("Cluster counts:")
print(X_scaled_df['Cluster'].value_counts())

### Step 6: Compute Silhouette Score, Inertia and Davies-Bouldin Index

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

silhouette = silhouette_score(X_scaled, kmeans.labels_)
inertia_val = kmeans.inertia_
db_index = davies_bouldin_score(X_scaled, kmeans.labels_)

print(f"Performance Metrics for k={k}:")
print(f"Silhouette Score: {silhouette:.3f}")
print(f"Inertia: {inertia_val:.2f}")
print(f"Davies-Bouldin Index: {db_index:.3f}")

In [ ]:
print("k | Silhouette | Davies-Bouldin")
for kk in range(2, 11):
    m = KMeans(n_clusters=kk, n_init=10, random_state=42).fit(X_scaled)
    print(kk, round(silhouette_score(X_scaled, m.labels_), 3),
          round(davies_bouldin_score(X_scaled, m.labels_), 3))

### Step 7: Visualise the clusters

In [ ]:
# (a) Two original features
plt.figure(figsize=(8, 5))
scatter = plt.scatter(X_scaled_df['github_repos'], X_scaled_df['expected_stipend'],
                      c=X_scaled_df['Cluster'], cmap='viridis')
plt.xlabel('GitHub Repositories (standardized)')
plt.ylabel('Expected Stipend (standardized)')
plt.title(f'Clusters Visualized (k={k})')
plt.colorbar(scatter, label='Cluster')
plt.show()

# (b) All seven features at once, projected to 2D with PCA
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=42)
P = pca.fit_transform(X_scaled)
plt.figure(figsize=(8, 5))
plt.scatter(P[:, 0], P[:, 1], c=X_scaled_df['Cluster'], cmap='viridis')
plt.xlabel('Principal Component 1'); plt.ylabel('Principal Component 2')
plt.title('Clusters in PCA space (all 7 features)')
plt.show()
print("Variance explained:", pca.explained_variance_ratio_.round(3))

### Step 8: Analyse cluster characteristics

In [ ]:
# Standardized cluster means
cluster_means = X_scaled_df.groupby('Cluster')[features].mean()
print("Cluster Characteristics (Mean Values, Standardized):")
print(cluster_means.round(2))

# Convert back to the original scale
X_original_with_clusters = pd.DataFrame(scaler.inverse_transform(X_scaled_df[features]),
                                        columns=features)
X_original_with_clusters['Cluster'] = X_scaled_df['Cluster']
original_means = X_original_with_clusters.groupby('Cluster')[features].mean()
print("\nCluster Characteristics (Mean Values, Original Scale):")
print(original_means.round(2))

# Percentage difference from the overall mean (the format RapidMiner/Weka-style outputs use)
overall_means = X_original_with_clusters[features].mean()
percent_diff = (original_means - overall_means) / overall_means * 100
print("\nPercentage Differences from Overall Dataset Mean:")
print(percent_diff.round(1))
print("\nOverall Dataset Means (Original Scale):")
print(overall_means.round(2))

### Step 9 (Extension): Validate the clusters against the hidden personas

In [ ]:
from sklearn.metrics import adjusted_rand_score

print("Adjusted Rand Index:", round(adjusted_rand_score(df['true_persona'], kmeans.labels_), 3))
print(pd.crosstab(df['true_persona'], kmeans.labels_,
                  rownames=['true_persona'], colnames=['Cluster']))

### Step 10 (Extension): Why scaling matters — clustering without standardisation

In [ ]:
kmeans_raw = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X.values)   # unscaled data

print("Share of total variance contributed by expected_stipend:",
      round(X.var()['expected_stipend'] / X.var().sum(), 6))
print("ARI vs true persona (unscaled):", round(adjusted_rand_score(df['true_persona'], kmeans_raw.labels_), 3))
print("Silhouette (unscaled):", round(silhouette_score(X.values, kmeans_raw.labels_), 3))